# Prompt Claude

Code for Single Claude requests.

In [17]:
import anthropic
import json
import yaml
from pathlib import Path
from typing import Dict, Any, List
from output_formats import output_json_explain, output_json_no_explain

In [18]:
def load_secrets(file_path: str = "secrets.json") -> dict:
    """Load API secrets from JSON file."""
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Secrets file not found: {file_path}")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)["CLAUDE_API_KEY"]
    
api_key = load_secrets("secrets.json")
client = anthropic.Anthropic(api_key=api_key)

In [19]:
# Load prompts from file
def load_prompts(file_path: str):
    """Load system and user prompts from a JSON or YAML file."""
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Prompt file not found: {file_path}")

    if path.suffix == ".json":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    elif path.suffix in [".yaml", ".yml"]:
        with open(path, "r", encoding="utf-8") as f:
            data = yaml.safe_load(f)
    else:
        raise ValueError("Unsupported file format. Use .json or .yaml")

    sys_prompt = data.get("system", "You are an expert annotator.")
    user_prompt = data["user"] 
    return sys_prompt, user_prompt

In [ ]:
input_type = "manual"
output_type = "json"
n_shot = "0shot"
model = "claude-3-5-haiku-20241022"
split = "test"
temperature = 1.0
# Safety: cap number of evaluated items to control cost/time
max_items = None # set to an int like 20 for a quick smoke test

sys_prompt, user_prompt = load_prompts(f"../../../prompts/{input_type}_{output_type}_{n_shot}.yaml")

data_path = "../../../data/reannotated_20250509_all.json"
results_path = f"../../../results/{input_type}_{output_type}_{n_shot}_{model}_{split}.json"

In [21]:
with open(data_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

# Filter data where 'split' == split
filtered_data = {k: v for k, v in data.items() if v.get('split') == split if k == "them-Wiki-neg-0163"}
number_examples = len(filtered_data)

In [22]:
# Prepare inputs
samples = list(filtered_data.items())
if max_items is not None:
    samples = samples[:max_items]

entries: List[Dict[str, Any]] = []

def parse_json_safely(s: str) -> Any:
    try:
        return json.loads(s)
    except Exception:
        # Try to extract JSON-looking substring
        import re
        matches = re.findall(r"\{[\s\S]*\}", s)
        for m in matches:
            try:
                return json.loads(m)
            except Exception:
                continue
        return {"raw": s}

# Helper: extract first tool_use block input from Anthropic response
def extract_tool_output(message):
    for block in getattr(message, "content", []):
        if getattr(block, "type", "") == "tool_use":
            # Claude tool call: block.name, block.input
            return getattr(block, "input", None)
    # Fallback: join any text blocks for debugging
    texts = []
    for block in getattr(message, "content", []):
        if getattr(block, "type", "") == "text":
            texts.append(getattr(block, "text", ""))
    return parse_json_safely("".join(texts)) if texts else None

for key, item in samples:
    text = item.get("text", "").strip()
    if not text:
        entries.append({
            "text_id": key,
            "input_text": text,
            "split": split,
            "output": {"error": "empty_text"}
        })
        continue

    user_content = user_prompt.format(text=text)

    try:
        resp = client.messages.create(
            model=model,
            max_tokens=1500,
            temperature=temperature,
            system=sys_prompt,
            messages=[{"role": "user", "content": user_content}],
            tools=[output_json_explain] if "explain" in output_type else [output_json_no_explain],
            tool_choice={"type": "tool", "name": "MoralisierungOutput"}
        )
        # Extract the structured tool output if present
        print(resp)
        result = extract_tool_output(resp)
        if result is None:
            result = {"error": "no_tool_use", "raw": [getattr(b, "text", None) for b in getattr(resp, "content", [])]}
    except Exception as e:
        result = {"error": str(e)}

    entry = {
        "text_id": key,
        "input_text": text,
        "split": split,
        "output": result
    }
    entries.append(entry)

# Write list[entry] to results_path
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(entries, f, ensure_ascii=False, indent=2)

len(entries)

Message(id='msg_01V8JFiMGVKE1PfNvcBoevDm', content=[ToolUseBlock(id='toolu_01MH3KnBAJsU13r2rYAH2umk', input={'moralisierung': {'moral_werte': [{'text': 'Fehler', 'moral_foundations_theory_kategorien': ['Betrug']}], 'forderung': 'Der Artikel soll korrigiert werden', 'enthaelt_moralisierung': True}, 'protagonisten': [{'text': 'Artikel', 'kategorie': 'Institution', 'rollen': ['Adressat:in', 'Malefizient:in']}, {'text': 'Verfasser des Artikels', 'kategorie': 'Individuum', 'rollen': ['Forderer:in']}]}, name='MoralisierungOutput', type='tool_use')], model='claude-3-5-haiku-20241022', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=8063, output_tokens=228, server_tool_use=None, service_tier='standard'))


1